# Federated Learning Client Selection — Comparison

Compares three client selection strategies:
1. **Random Selection** — baseline, picks k clients uniformly at random
2. **DivFL (Greedy Diverse)** — selects diverse clients by gradient dissimilarity
3. **S-FedAvg (Shapley)** — selects clients probabilistically via Shapley value scores computed on a held-out validation set

In [ ]:
import torch
import copy
from torch.utils.data import DataLoader, Subset
import numpy as np

from utils.data import load_data, create_clients, get_client_loader
from client.client import Client
from models.cnn import CNN
from selection.random_sel import random_selection
from selection.greedy_divfl import greedy_divfl
from experiments.run_experiment import run_federated, run_federated_divfl, run_sfedavg
from utils.metrics import accuracy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# =========================
# Load dataset
# =========================
train_dataset, test_dataset = load_data()

# =========================
# Create server validation set (1000 samples from test set)
# Used by S-FedAvg to evaluate Shapley contributions
# =========================
val_indices = np.random.choice(len(test_dataset), size=1000, replace=False)
val_subset = Subset(test_dataset, val_indices)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

print(f"Validation set size: {len(val_subset)} samples")

In [ ]:
# =========================
# Create client splits
# =========================
client_indices = create_clients(train_dataset, num_clients=10)

clients = {}
for i in client_indices:
    loader = get_client_loader(train_dataset, client_indices[i], batch_size=32)
    clients[i] = Client(loader, device)

print(f"Created {len(clients)} clients")

In [ ]:
# =========================
# Initialize base model (shared starting point for all 3 runs)
# =========================
base_model = CNN().to(device)

model_random = copy.deepcopy(base_model)
model_divfl  = copy.deepcopy(base_model)
model_sfedavg = copy.deepcopy(base_model)

In [ ]:
# =========================
# TRAIN WITH RANDOM SELECTION
# =========================
print("\n===== Training with RANDOM Selection =====")

global_model_random = run_federated(
    model_random,
    clients,
    random_selection,
    rounds=20,
    k=5
)

acc_random = accuracy(global_model_random, test_dataset, device)
print(f"Random Selection Accuracy: {acc_random:.4f}")

In [ ]:
# =========================
# TRAIN WITH GREEDY DIVFL
# =========================
print("\n===== Training with GREEDY DivFL =====")

global_model_divfl = run_federated_divfl(
    model_divfl,
    clients,
    greedy_divfl,
    rounds=20,
    k=5
)

acc_divfl = accuracy(global_model_divfl, test_dataset, device)
print(f"DivFL Accuracy: {acc_divfl:.4f}")

In [ ]:
# =========================
# TRAIN WITH S-FedAvg (Shapley)
# =========================
print("\n===== Training with S-FedAvg (Shapley Value Selection) =====")
print("Note: Computing Shapley values each round. T=5 permutations for speed.")

global_model_sfedavg = run_sfedavg(
    model_sfedavg,
    clients,
    val_loader=val_loader,
    rounds=20,
    k=5,
    T=5          # increase T for more accurate Shapley estimates (slower)
)

acc_sfedavg = accuracy(global_model_sfedavg, test_dataset, device)
print(f"S-FedAvg Accuracy: {acc_sfedavg:.4f}")

In [ ]:
# =========================
# FINAL COMPARISON
# =========================
print("\n" + "="*50)
print("          FINAL ACCURACY COMPARISON")
print("="*50)
print(f"  Random Selection   : {acc_random:.4f}")
print(f"  DivFL              : {acc_divfl:.4f}")
print(f"  S-FedAvg (Shapley) : {acc_sfedavg:.4f}")
print("="*50)

best_acc = max(acc_random, acc_divfl, acc_sfedavg)
if best_acc == acc_sfedavg:
    print("✅ S-FedAvg (Shapley) performed best!")
elif best_acc == acc_divfl:
    print("✅ DivFL performed best!")
else:
    print("⚠️ Random performed best — check data distribution or increase rounds.")